In [ ]:
# Example 1: top-level public API surface.
import langchain_groq
from langchain_groq import ChatGroq

# ChatGroq should be re-exported at top-level
assert hasattr(langchain_groq, 'ChatGroq')
assert langchain_groq.ChatGroq is ChatGroq
print(f"ChatGroq class: {ChatGroq}")
print(f"ChatGroq module: {ChatGroq.__module__}")

In [ ]:
# Example 2: ChatGroq instantiation (no network call, dummy API key).
# Note: the Groq API rejects temperature=0 exactly, so langchain_groq's validator
# coerces 0.0 to the API minimum 1e-8 — assert that documented coercion.
from langchain_groq import ChatGroq

chat = ChatGroq(
    model='llama-3.1-8b-instant',
    temperature=0.0,
    groq_api_key='gsk_dummy_key_for_test_only',
)
print(f"model_name: {chat.model_name}")
print(f"temperature: {chat.temperature}")
print(f"type: {type(chat).__name__}")
assert chat.model_name == 'llama-3.1-8b-instant'
assert chat.temperature == 1e-8  # 0.0 is coerced to the API-minimum 1e-8

In [ ]:
# Example 3: ChatGroq inherits from langchain_core BaseChatModel.
from langchain_groq import ChatGroq
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.runnables import Runnable

assert issubclass(ChatGroq, BaseChatModel)
assert issubclass(ChatGroq, Runnable)

chat = ChatGroq(model='llama-3.1-8b-instant', groq_api_key='gsk_dummy')
# _llm_type identifier the LangChain framework uses
print(f"_llm_type: {chat._llm_type}")
assert isinstance(chat._llm_type, str)

In [ ]:
# Example 4: message formatting helpers + prompt template integration.
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

chat = ChatGroq(model='llama-3.1-8b-instant', groq_api_key='gsk_dummy')

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', '{question}'),
])

# Build a pipeline without invoking it (no API call)
chain = prompt | chat
print(f"chain type: {type(chain).__name__}")

messages = [SystemMessage(content='sys'), HumanMessage(content='hi')]
assert len(messages) == 2

In [ ]:
# Example 5: tool-binding interface (bind_tools) — surface check only, no API call.
from langchain_groq import ChatGroq
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

chat = ChatGroq(model='llama-3.1-8b-instant', groq_api_key='gsk_dummy')
assert hasattr(chat, 'bind_tools'), 'ChatGroq must support bind_tools'

bound = chat.bind_tools([add])
print(f"bound type: {type(bound).__name__}")
assert bound is not None
print('OK langchain_groq smoke tests')